In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os
load_dotenv()

if os.environ['GOOGLE_API_KEY']:
    print("Google API Key is set.")
else:
    raise ValueError("Google API Key is not set.")

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

### **First Graph**

### **Step-1: Define The Schema**

In [ ]:
from typing import TypedDict, List, Annotated
from langgraph.graph.message import add_messages

class graph_schema(TypedDict):
    messages: Annotated[List, add_messages]

### **Step-2: Create The Node Functions**

In [ ]:
def welcome(schema:graph_schema) -> graph_schema:

    curr_message = schema['messages']

    response = llm.invoke(curr_message).content

    schema['messages'] = f"Your message was {curr_message}. Here's my response:  {response}"

    return schema

### **Step-3: Create The State Graph**

In [ ]:
###
from langgraph.graph import StateGraph, START, END

graph = StateGraph(graph_schema)

# Adding Nodes
graph.add_node("welcome", welcome)

# Adding Edges
graph.add_edge(START, "welcome")
graph.add_edge("welcome", END)

### **Step-4 Compile The Graph With Checkpoint**

In [ ]:
from IPython.display import Image, display
from langgraph.checkpoint.memory import InMemorySaver  

checkpoint = InMemorySaver()

memory_graph = graph.compile(checkpointer=checkpoint)

# You could see the errors with the below command
Image(memory_graph.get_graph().draw_mermaid_png())

### **Step-5: Run The Graph With Config**

In [ ]:
response = memory_graph.invoke(
    {"messages":"What is my name?"},
    {'configurable':{'thread_id': 'Shrinath'}}
)

for message in response['messages']:
    message.pretty_print()